Here is the easiest way to separate them in your head:

PyObject is the Instance (The physical building).

PyTypeObject is the Class (The architectural blueprint).

Here is exactly how they differ in the CPython source code.

1. PyObject (The Instance)
This is the absolute base structure for every single piece of data you create in Python. Whether you make a string ("hello"), a number (42), or your custom instance (c = Child()), the C-engine builds a PyObject in memory.

It is incredibly tiny. It only holds two things:

C
typedef struct _object {
    Py_ssize_t ob_refcnt;        // How many variables are looking at me? (Garbage Collection)
    struct _typeobject *ob_type; // WHO BUILT ME? (Pointer to the blueprint)
} PyObject;
Notice what is missing: There are no methods here. There is no MRO here. It is just a reference count and a pointer to its class.

2. PyTypeObject (The Blueprint)
This is the massive C-struct that defines a Class. When you write class Child(Parent):, the engine builds one of these. It contains the actual rules, methods, and inheritance maps.

It is huge (over 100 fields in C), but here are the heavy hitters:

C
typedef struct _typeobject {
    PyObject_VAR_HEAD            // (Wait, what is this? See the mind-bender below)
    const char *tp_name;         // The name of the class (e.g., "Child")
    Py_ssize_t tp_basicsize;     // How much RAM does an instance need?
    
    // The Pointers we talked about:
    struct _dictobject *tp_dict; // The dictionary holding the methods (like 'greet')
    PyObject *tp_bases;          // Pointer to Parent classes
    PyObject *tp_mro;            // Pointer to the full search path array
    
    // C-level function pointers for dunder methods:
    newfunc tp_new;              // The C-code for __new__
    initproc tp_init;            // The C-code for __init__
    reprfunc tp_str;             // The C-code for __str__
} PyTypeObject;
The Mind-Bender: Why Classes are Objects
Look closely at the very first line inside PyTypeObject: PyObject_VAR_HEAD.

This is a C macro. When the C-compiler runs, it literally expands that macro and drops a complete PyObject struct right into the top of the PyTypeObject.

This is why "Everything in Python is an object."

Because PyTypeObject physically contains a PyObject at its core, a Class has its own reference count (ob_refcnt) and its own pointer to who built it (ob_type).

Who builds a Class? The metaclass type. So if you follow the ob_type pointer of the Child class blueprint, it points straight to the type blueprint.

The Ultimate Summary
PyObject: "I am data. Here is my memory count, and here is a pointer to the blueprint that explains how I work."

PyTypeObject: "I am a blueprint. Here is the MRO, the dictionary of methods, and the C-functions needed to govern the data."